# ConvNext with Entropy Filtering and Ordinal Loss

This notebook trains an ConvNext model with:
- **Entropy filtering**: Removes high-entropy samples for cleaner training
- **Ordinal loss**: Leverages the ordinal nature of ISUP grades (0 < 1 < 2 < 3 < 4 < 5)

In [1]:
from torch import optim
from torchvision.models import convnext_tiny, ConvNeXt_Tiny_Weights
import torch
import random
import numpy as np
import torch.nn as nn
import albumentations as Albu
import pandas as pd
from torch.utils.data import DataLoader
from torch.utils.data.sampler import RandomSampler, SequentialSampler
from warmup_scheduler import GradualWarmupScheduler
from sklearn.metrics import accuracy_score, cohen_kappa_score, f1_score, recall_score, precision_score
from tqdm import tqdm
import os
import sys
import optuna
from optuna.pruners import MedianPruner
sys.path.append('../../../')
from utils.dataset import PandasDataset
from utils.models import EfficientNetApi


## Configuration

In [2]:
# Training parameters
seed = 42
batch_size = 3
num_workers = 4
output_classes = 5  # For ordinal encoding: ISUP 0-5 → 5 thresholds
init_lr = 3e-4
warmup_factor = 2
warmup_epochs = 1
n_epochs = 50
dropout_rate = 0.3
patience = 7

# Optuna Configuration
OPTUNA_SUBSET_FRAC = 0.2
OPTUNA_BATCH_SIZE = batch_size
N_OPTUNA_EPOCHS = 10
N_TRIALS = 30
STUDY_DB = 'sqlite:///../logs/convnext-ordinal-optuna.db'
STUDY_NAME = 'convnext-ordinal-optuna'

# Device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Set seeds
torch.manual_seed(seed)
random.seed(seed)
np.random.seed(seed)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(seed)

# Paths
ROOT_DIR = '../../..'
data_dir = '../../../..'
images_dir = os.path.join(data_dir, 'tiles')

# Output paths
os.makedirs('../logs', exist_ok=True)
os.makedirs('../models', exist_ok=True)
model_path = 'models/convnext-optuna.pth'
log_path = 'logs/convnext-optuna.txt'


Using device: cpu


/home/woshington/Projects/Doutorado/repo/.venv/lib/python3.12/site-packages/torch/cuda/__init__.py:182: UserWarning: CUDA initialization: Unexpected error from cudaGetDeviceCount(). Did you run some cuda functions before calling NumCudaDevices() that might have already set an error? Error 804: forward compatibility was attempted on non supported HW (Triggered internally at /pytorch/c10/cuda/CUDAFunctions.cpp:109.)
  return torch._C._cuda_getDeviceCount() > 0


## Ordinal Loss Function

Ordinal regression treats the problem as a series of binary classifications:
- For ISUP grade k, we predict k binary labels: [1,1,...,1,0,0,...,0]
- Example: ISUP 2 → [1, 1, 0, 0, 0] (grade > 0, grade > 1, but not > 2, > 3, > 4)

In [3]:
import torch
import torch.nn as nn
import torch.nn.functional as F


class OrdinalFocalRegressionLoss(nn.Module):

    def __init__(
        self,
        alpha: float = 0.25,
        gamma: float = 2.0,
        ordinal_weight: float = 0.2,
        reduction: str = 'mean'
    ):
        super().__init__()

        self.alpha = alpha
        self.gamma = gamma
        self.ordinal_weight = ordinal_weight
        self.reduction = reduction

    def forward(
        self,
        logits: torch.Tensor,
        targets: torch.Tensor
    ) -> torch.Tensor:

        # numerical stability under AMP
        logits = logits.float()
        targets = targets.float()

        probs = torch.sigmoid(logits)

        # BCE
        bce = F.binary_cross_entropy_with_logits(
            logits,
            targets,
            reduction='none'
        )

        # focal term
        p_t = probs * targets + (1 - probs) * (1 - targets)

        loss = self.alpha * ((1 - p_t) ** self.gamma) * bce

        # reduction
        if self.reduction == 'mean':
            focal_loss = loss.mean()
        elif self.reduction == 'sum':
            focal_loss = loss.sum()
        else:
            focal_loss = loss

        # ordinal penalty
        expected_class = probs.sum(dim=1)
        target_class = targets.sum(dim=1)

        max_class = logits.shape[1]

        ordinal_loss = (
            (expected_class - target_class) ** 2
        ).mean() / (max_class ** 2)

        # final loss
        total_loss = focal_loss + (
            self.ordinal_weight * ordinal_loss
        )

        return total_loss


def decode_ordinal_predictions(logits):
    """
    Convert ordinal predictions back to class labels.

    Args:
        logits: (batch_size, num_classes) - raw model outputs

    Returns:
        Predicted ISUP grades (0-5)
    """
    # Apply sigmoid to get probabilities
    probs = torch.sigmoid(logits)

    # Sum probabilities > 0.5 to get predicted grade
    predictions = (probs > 0.5).sum(dim=1)

    return predictions

## Load Data with Entropy Filtering

In [4]:
# Load original training data
df_train_ = pd.read_csv(f"{ROOT_DIR}/data/train_5fold.csv")
print(f"Original records: {len(df_train_)}")

# Load entropy filtered samples
df_entropy = pd.read_csv(f"{ROOT_DIR}/data/entropy.csv")
print(f"High-entropy samples to remove: {len(df_entropy)}")

df_entropy_sorted = df_entropy.sort_values(by='difficulty_score', ascending=False).reset_index(drop=True)

print(df_entropy_sorted.head())
n_remove = int(len(df_entropy) * 0.2)

df_entropy_top = df_entropy_sorted.head(n_remove)

# Maneira Correta
print("max_entropy", df_entropy_top["isup_grade"].value_counts())
# Filter out high-entropy samples
df_train_ = df_train_[~df_train_['image_id'].isin(df_entropy_top['image_id'])].reset_index(drop=True)
print(f"Filtered records: {len(df_train_)}")

# Clean column names
df_train_.columns = df_train_.columns.str.strip()

# Split by fold (fold 3 for validation)
train_indexes = np.where(df_train_['fold'] != 3)[0]
valid_indexes = np.where(df_train_['fold'] == 3)[0]

df_train = df_train_.loc[train_indexes].reset_index(drop=True)
df_val = df_train_.loc[valid_indexes].reset_index(drop=True)

# Load test data
df_test = pd.read_csv(f"{ROOT_DIR}/data/test.csv")

def remove_nonexistent_images(df, images_dir):
    """
    Remove rows from df where the image does not exist in images_dir.
    """
    image_ids = df['image_id'].apply(lambda x: os.path.join(images_dir, f"{x}.png"))
    existent_images = [os.path.isfile(path) for path in image_ids]
    df = df[existent_images]
    return df

df_train = remove_nonexistent_images(df_train, images_dir)
df_val = remove_nonexistent_images(df_val, images_dir)
df_test = remove_nonexistent_images(df_test, images_dir)

print(f"\nTrain: {len(df_train)} samples")
print(f"Validation: {len(df_val)} samples")
print(f"Test: {len(df_test)} samples")

# Check class distribution
print(f"\nTrain class distribution:")
print(df_train['isup_grade'].value_counts().sort_index())

Original records: 9024
High-entropy samples to remove: 903
                           image_id data_provider  isup_grade gleason_score  \
0  e0f8b96960ada384a00e493545f783da       radboud           5           5+5   
1  c3f6dfc5c801b1f2aed4c9e318bd015d       radboud           5           5+5   
2  35c7912e941c9bf21594deeda6c891e2       radboud           3           4+3   
3  a4514f8a6800bf122ae746c4f573ee6f       radboud           5           5+5   
4  0a8c2bda6e00a040372185ccd9a3c4ab    karolinska           5           4+5   

   fold             image_id_from_results  true_label  pred_b0  entropy_b0  \
0     1  e0f8b96960ada384a00e493545f783da         5.0      3.0    0.646624   
1     2  c3f6dfc5c801b1f2aed4c9e318bd015d         5.0      0.0    0.114379   
2     0  35c7912e941c9bf21594deeda6c891e2         3.0      4.0         NaN   
3     0  a4514f8a6800bf122ae746c4f573ee6f         5.0      4.0    0.327839   
4     0  0a8c2bda6e00a040372185ccd9a3c4ab         5.0      4.0    0.319212  

## Data Augmentation

In [5]:
train_transforms = Albu.Compose([
    Albu.Transpose(p=0.5),
    Albu.VerticalFlip(p=0.5),
    Albu.HorizontalFlip(p=0.5),
    Albu.RandomBrightnessContrast(p=0.3),
    Albu.HueSaturationValue(p=0.2),
])

val_transforms = None

## Create Datasets and DataLoaders

In [6]:
from torch.utils.data import SequentialSampler

# Subset aleatório fixo para o Optuna
rng_optuna = np.random.default_rng(seed)
optuna_idx = rng_optuna.choice(len(df_train), size=int(len(df_train) * OPTUNA_SUBSET_FRAC), replace=False)
df_optuna = df_train.iloc[optuna_idx].reset_index(drop=True)

print(f'Optuna subset: {len(df_optuna)} imgs ({OPTUNA_SUBSET_FRAC:.0%})')

optuna_train_ds = PandasDataset(images_dir, df_optuna, transforms=train_transforms, format="png")
optuna_val_ds = PandasDataset(images_dir, df_val, transforms=val_transforms, format="png")

optuna_train_loader = DataLoader(
    optuna_train_ds,
    batch_size=OPTUNA_BATCH_SIZE,
    num_workers=num_workers,
    sampler=RandomSampler(optuna_train_ds),
    pin_memory=True,
    persistent_workers=True,
    drop_last=True
)

optuna_val_loader = DataLoader(
    optuna_val_ds,
    batch_size=OPTUNA_BATCH_SIZE * 2,
    num_workers=num_workers,    
    pin_memory=True,
    persistent_workers=True,
)

print(f"Optuna Train batches: {len(optuna_train_loader)}")
print(f"Optuna Validation batches: {len(optuna_val_loader)}")


Optuna subset: 1414 imgs (20%)
Optuna Train batches: 471
Optuna Validation batches: 295


## Model Setup

In [7]:
from utils.models import ConvNeXtApi

def build_model(dropout_):
    load_model = convnext_tiny(weights=ConvNeXt_Tiny_Weights.DEFAULT)
    model = ConvNeXtApi(model=load_model, output_dimensions=output_classes, dropout_rate=dropout_)
    return model.to(device)


## Optimizer and Scheduler

## Training and Validation Functions

In [ ]:
def training_step(model, dataloader, optimizer, device, loss_fn, scaler, accumulation_steps=2):
    """
    Perform one training epoch.
    """
    model.train()
    train_loss = []
    
    bar_progress = tqdm(dataloader, desc="Training", dynamic_ncols=True)
    optimizer.zero_grad()
    
    for step, (batch_data, batch_targets, _) in enumerate(bar_progress):
        batch_data = batch_data.to(device)
        batch_targets = batch_targets.to(device)  # Already in ordinal format from PandasDataset
        
        with torch.autocast(device_type='cuda', dtype=torch.float16):
            logits = model(batch_data)
            loss = loss_fn(logits, batch_targets) / accumulation_steps
        
        scaler.scale(loss).backward()

        if (step + 1) % accumulation_steps == 0 or (step + 1) == len(dataloader):
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad()
        
        train_loss.append((loss.detach() * accumulation_steps).item())
        smooth_loss = sum(train_loss[-100:]) / min(len(train_loss), 100)

        if step % 10 == 0:
            bar_progress.set_postfix({
                'loss': f'{train_loss[-1]:.5f}',
                'smooth': f'{smooth_loss:.5f}'
            })

    return train_loss

def validation_step(model, dataloader, device, loss_fn):
    """
    Perform validation.
    """
    model.eval()
    
    total_loss = 0.0
    num_batches = 0
    all_preds = []
    all_targets = []
    
    bar_progress = tqdm(dataloader, desc="Validation", dynamic_ncols=True)

    
    with torch.no_grad():
        for batch_data, batch_targets, _ in bar_progress:
            batch_data = batch_data.to(device)
            batch_targets_ordinal = batch_targets.to(device)  # Already ordinal

            with torch.autocast(device_type='cuda', dtype=torch.float16):
                logits = model(batch_data)
                loss = loss_fn(logits, batch_targets_ordinal)
            
            probs = torch.sigmoid(logits)
            predictions = (probs > 0.5).sum(dim=1)
            targets_class = batch_targets.sum(dim=1).long()

            # Acumula em tensores na CPU — sem conversão prematura
            all_preds.append(predictions.cpu())
            all_targets.append(targets_class.cpu())

            total_loss += loss.item()
            num_batches += 1
    
    all_preds = torch.cat(all_preds).numpy()
    all_targets = torch.cat(all_targets).numpy()
    
    return {
        'val_loss': total_loss / num_batches,
        'val_acc': accuracy_score(all_targets, all_preds),
        'val_kappa': cohen_kappa_score(all_targets, all_preds, weights='quadratic'),
        'val_f1': f1_score(all_targets, all_preds, average='macro', zero_division=0),
        'val_recall': recall_score(all_targets, all_preds, average='macro', zero_division=0),
        'val_precision': precision_score(all_targets, all_preds, average='macro', zero_division=0),
    }

## Training Loop

In [ ]:
def objective(trial: optuna.Trial) -> float:
    lr              = trial.suggest_float('lr',              1e-5, 5e-3, log=True)
    dropout_rate    = trial.suggest_float('dropout_rate',    0.2,  0.6)
    focal_gamma     = trial.suggest_float('focal_gamma',     0.5,  4.0)
    focal_alpha     = trial.suggest_float('focal_alpha',     0.1,  0.5)
    batch_size      = trial.suggest_categorical('batch_size', [2, 4])  # removed 8
    weight_decay    = trial.suggest_float('weight_decay',    1e-6, 1e-2, log=True)

    model = build_model(dropout_=dropout_rate)
    loss_function = OrdinalFocalRegressionLoss(
        alpha=focal_alpha, gamma=focal_gamma
    )
    
    optimizer = optim.Adam(model.parameters(), lr=lr / warmup_factor)
    scheduler_cosine = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, N_OPTUNA_EPOCHS - warmup_epochs)
    scheduler = GradualWarmupScheduler(
        optimizer,
        multiplier=warmup_factor,
        total_epoch=warmup_epochs,
        after_scheduler=scheduler_cosine
    )

    scaler = torch.amp.GradScaler(device='cuda')
    best_kappa = 0.0
    
    for epoch in range(N_OPTUNA_EPOCHS):
        training_step(model, optuna_train_loader, optimizer, device, loss_function, scaler, accumulation_steps=2)
        metrics = validation_step(model, optuna_val_loader, device, loss_function)
        kappa = metrics['val_kappa']
        
        scheduler.step()

        trial.report(kappa, epoch)
        if trial.should_prune():
            del model
            torch.cuda.empty_cache()
            raise optuna.exceptions.TrialPruned()

        best_kappa = max(best_kappa, kappa)

    del model
    torch.cuda.empty_cache()
    
    return best_kappa


In [10]:
optuna.logging.set_verbosity(optuna.logging.WARNING)

study = optuna.create_study(
    study_name=STUDY_NAME,
    direction='maximize',
    pruner=MedianPruner(n_startup_trials=5, n_warmup_steps=2),
    storage=STUDY_DB,
    load_if_exists=True,
)

print("Starting Optuna optimization...")
study.optimize(objective, n_trials=N_TRIALS)

print("\n=========================================")
print("Best trial:")
best = study.best_trial
print(f"  Value (Kappa): {best.value:.4f}")
print("  Params:")
for key, value in best.params.items():
    print(f"    {key}: {value}")


Starting Optuna optimization...


/home/woshington/Projects/Doutorado/repo/.venv/lib/python3.12/site-packages/torch/amp/grad_scaler.py:136: UserWarning: torch.cuda.amp.GradScaler is enabled, but CUDA is not available.  Disabling.
  warnings.warn(
Training:   0%|          | 0/471 [00:00<?, ?it/s]/home/woshington/Projects/Doutorado/repo/.venv/lib/python3.12/site-packages/torch/utils/data/dataloader.py:666: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)
/home/woshington/Projects/Doutorado/repo/.venv/lib/python3.12/site-packages/torch/amp/autocast_mode.py:266: UserWarning: User provided device_type of 'cuda', but CUDA is not available. Disabling
  warnings.warn(
Training:   0%|          | 0/471 [00:07<?, ?it/s]
[W 2026-05-21 09:09:30,130] Trial 0 failed with parameters: {'lr': 6.957814959900319e-05, 'dropout_rate': 0.40034016271449996, 'focal_gamma': 1.333707543359906, 'focal_alpha': 0.34859641224277693, 'batch_size': 4, 'wei

NameError: name 'scaler' is not defined

In [ ]:
importances = optuna.importance.get_param_importances(study)
print('Importância dos hiperparâmetros:')
for param, imp in importances.items():
    print(f'  {param}: {imp:.4f}')


In [ ]:
from optuna import visualization as optvis

fig = optvis.plot_optimization_history(study)
fig.update_layout(title='Histórico de Otimização — Kappa por Trial', height=450)
fig.show()


In [ ]:
print(f'Storage: {STUDY_DB}')
print(f'Study  : {STUDY_NAME}')
print('\nPara abrir o dashboard, execute no terminal:')
print(f'  optuna-dashboard {STUDY_DB}')


## 5-Fold Cross-Validation

Usa os hiperparâmetros ótimos do Optuna e treina um modelo por fold.
Cada fold usa os demais 4 como treino e o fold atual como validação.

| Fold | Treino | Validação |
|------|--------|-----------|
| 0 | folds 1–4 | fold 0 |
| 1 | folds 0,2–4 | fold 1 |
| … | … | … |
| 4 | folds 0–3 | fold 4 |

Ao final: média ± std do QWK entre folds + ensemble dos 5 checkpoints no test set.

In [ ]:
import copy

# ── Hiperparâmetros: Optuna best ou fallback ───────────────────────────
if 'study' in dir() and study.best_trial is not None:
    bp = study.best_params
else:
    bp = {
        'lr': 1.1e-4, 'dropout_rate': 0.3, 'focal_gamma': 2.0,
        'focal_alpha': 0.25, 'batch_size': 4, 'weight_decay': 1e-5,
    }
print('Hiperparâmetros usados no CV:')
for k, v in bp.items():
    print(f'  {k}: {v}')

N_FOLDS      = 5
CV_PATIENCE  = patience   # mesmo do config global
CV_EPOCHS    = n_epochs   # mesmo do config global

fold_metrics = []         # {'fold', 'best_kappa', 'best_acc', 'best_f1', 'history'}
fold_ckpts   = {}         # fold → state_dict (em CPU)

for fold in range(N_FOLDS):
    print(f'\n{"="*55}')
    print(f'  FOLD {fold}')
    print(f'{"="*55}')

    df_tr_f = df_train_[df_train_['fold'] != fold].reset_index(drop=True)
    df_vl_f = df_train_[df_train_['fold'] == fold].reset_index(drop=True)
    df_tr_f = remove_nonexistent_images(df_tr_f, images_dir)
    df_vl_f = remove_nonexistent_images(df_vl_f, images_dir)
    print(f'  Treino: {len(df_tr_f)}  |  Val: {len(df_vl_f)}')

    # Dataloaders
    bs = bp['batch_size']
    tr_ds = PandasDataset(images_dir, df_tr_f, transforms=train_transforms, format='png')
    vl_ds = PandasDataset(images_dir, df_vl_f, transforms=val_transforms, format='png')
    tr_dl = DataLoader(tr_ds, batch_size=bs, num_workers=num_workers,
                       sampler=RandomSampler(tr_ds),
                       pin_memory=True, drop_last=True)
    vl_dl = DataLoader(vl_ds, batch_size=bs * 2, num_workers=num_workers,
                       sampler=SequentialSampler(vl_ds), pin_memory=True)

    # Modelo, loss, otimizador
    model   = build_model(dropout_=bp['dropout_rate'])
    loss_fn = OrdinalFocalRegressionLoss(
        alpha=bp['focal_alpha'], gamma=bp['focal_gamma'])
    optimizer = optim.Adam(model.parameters(),
                           lr=bp['lr'] / warmup_factor,
                           weight_decay=bp['weight_decay'])
    sched_cos = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer, CV_EPOCHS - warmup_epochs)
    scheduler = GradualWarmupScheduler(
        optimizer, multiplier=warmup_factor,
        total_epoch=warmup_epochs, after_scheduler=sched_cos)
    scaler = torch.amp.GradScaler(device=str(device).split(':')[0])

    # Loop de treinamento
    best_kappa, best_acc, best_f1 = 0.0, 0.0, 0.0
    best_state    = None
    no_improve    = 0
    history       = []

    for epoch in range(CV_EPOCHS):
        tr_loss = training_step(model, tr_dl, optimizer, device, loss_fn, scaler)
        m       = validation_step(model, vl_dl, device, loss_fn)
        scheduler.step()

        kappa = m['val_kappa']
        history.append({
            'epoch': epoch + 1,
            'train_loss': float(np.mean(tr_loss)),
            **{k: float(v) for k, v in m.items()},
        })
        print(f'  ep {epoch+1:02d} | loss={np.mean(tr_loss):.4f} '
              f'| kappa={kappa:.4f} | acc={m["val_acc"]*100:.2f}%')

        if kappa > best_kappa:
            best_kappa  = kappa
            best_acc    = m['val_acc']
            best_f1     = m['val_f1']
            best_state  = copy.deepcopy(model.state_dict())
            no_improve  = 0
            # Salva checkpoint do fold
            torch.save(best_state, f'models/convnext-fold{fold}.pth')
        else:
            no_improve += 1
            if no_improve >= CV_PATIENCE:
                print(f'  Early stopping (patience={CV_PATIENCE})')
                break

    fold_ckpts[fold] = best_state
    fold_metrics.append({
        'fold': fold, 'best_kappa': best_kappa,
        'best_acc': best_acc, 'best_f1': best_f1,
        'history': history,
    })
    print(f'  → FOLD {fold} melhor QWK={best_kappa:.4f}')

    del model, best_state
    torch.cuda.empty_cache()

# ── Resultados por fold ───────────────────────────────────────────────
kappas = np.array([r['best_kappa'] for r in fold_metrics])
accs   = np.array([r['best_acc']   for r in fold_metrics])
f1s    = np.array([r['best_f1']    for r in fold_metrics])

print(f'\n{"="*55}')
print('  RESULTADOS 5-FOLD CV')
print(f'{"="*55}')
print(f'  {"Fold":<6} {"QWK":>8} {"Acc":>8} {"F1":>8}')
for r in fold_metrics:
    print(f'  {r["fold"]:<6} {r["best_kappa"]:>8.4f} '
          f'{r["best_acc"]*100:>7.2f}% {r["best_f1"]:>8.4f}')
print(f'  {"Mean":<6} {kappas.mean():>8.4f} '
      f'{accs.mean()*100:>7.2f}% {f1s.mean():>8.4f}')
print(f'  {"Std":<6} {kappas.std():>8.4f} '
      f'{accs.std()*100:>7.2f}% {f1s.std():>8.4f}')


In [ ]:
import matplotlib.pyplot as plt

# ── Curvas de treinamento por fold ────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
colors = plt.cm.tab10(np.linspace(0, 0.5, N_FOLDS))

for i, r in enumerate(fold_metrics):
    hist   = r['history']
    epochs = [h['epoch'] for h in hist]
    kap    = [h['val_kappa'] for h in hist]
    loss   = [h['train_loss'] for h in hist]
    label  = f'Fold {r["fold"]} (best={r["best_kappa"]:.4f})'
    axes[0].plot(epochs, kap,  color=colors[i], label=label)
    axes[1].plot(epochs, loss, color=colors[i], label=f'Fold {r["fold"]}')

axes[0].set_title('QWK de validação por fold', fontsize=11)
axes[0].set_xlabel('Época'); axes[0].set_ylabel('QWK')
axes[0].axhline(kappas.mean(), color='black', linestyle='--',
                label=f'Mean={kappas.mean():.4f}')
axes[0].legend(fontsize=8); axes[0].grid(alpha=0.35)

axes[1].set_title('Train loss por fold', fontsize=11)
axes[1].set_xlabel('Época'); axes[1].set_ylabel('Loss')
axes[1].legend(fontsize=8); axes[1].grid(alpha=0.35)

plt.suptitle('ConvNeXt-Tiny — 5-Fold CV', fontsize=13)
plt.tight_layout()
plt.savefig('logs/convnext-cv-training.png', dpi=200, bbox_inches='tight')
plt.show()

# ── Bar chart do QWK por fold ─────────────────────────────────────────
fig, ax = plt.subplots(figsize=(7, 4))
bar_c = ['#3498db'] * N_FOLDS
bars = ax.bar([f'Fold {r["fold"]}' for r in fold_metrics],
              kappas, color=bar_c, edgecolor='white')
ax.axhline(kappas.mean(), color='red', linestyle='--',
           label=f'Mean={kappas.mean():.4f} ± {kappas.std():.4f}')
for bar, v in zip(bars, kappas):
    ax.text(bar.get_x() + bar.get_width()/2, v + 0.001,
            f'{v:.4f}', ha='center', fontsize=9)
ax.set_ylabel('QWK (validação)'); ax.set_ylim(0, 1)
ax.set_title('ConvNeXt-Tiny — QWK por Fold', fontsize=11)
ax.legend(); ax.grid(axis='y', alpha=0.35)
plt.tight_layout()
plt.savefig('logs/convnext-cv-kappa-per-fold.png', dpi=200, bbox_inches='tight')
plt.show()


## Avaliação no Test Set — Ensemble dos 5 Folds

Carrega os 5 checkpoints e faz a média das probabilidades (ensemble).

In [ ]:
from sklearn.metrics import (classification_report, confusion_matrix,
                             accuracy_score, cohen_kappa_score,
                             f1_score, recall_score, precision_score)
import seaborn as sns

# ── DataLoader do test set ─────────────────────────────────────────────
test_ds = PandasDataset(images_dir, df_test, transforms=val_transforms, format='png')
test_dl = DataLoader(test_ds, batch_size=bp['batch_size'] * 2,
                     num_workers=num_workers,
                     sampler=SequentialSampler(test_ds), pin_memory=True)

def infer_probs(state_dict, dataloader, device):
    """Carrega state_dict num modelo e retorna sigmoid probs + targets."""
    m = build_model(dropout_=bp['dropout_rate'])
    m.load_state_dict(state_dict)
    m = m.to(device).eval()
    probs_all, tgts_all = [], []
    with torch.no_grad():
        for bx, by, _ in dataloader:
            p = torch.sigmoid(m(bx.to(device))).cpu()
            probs_all.append(p)
            tgts_all.append(by.sum(1).long())
    del m; torch.cuda.empty_cache()
    return torch.cat(probs_all).numpy(), torch.cat(tgts_all).numpy()

# ── Coleta probs de cada fold ──────────────────────────────────────────
print('Inferência dos 5 folds no test set...')
fold_probs = []
test_targets_cv = None

for fold in range(N_FOLDS):
    # Tenta memória primeiro; fallback para disco
    state = fold_ckpts.get(fold)
    if state is None:
        ckpt_path = f'models/convnext-fold{fold}.pth'
        state = torch.load(ckpt_path, weights_only=True, map_location='cpu')

    probs, tgts = infer_probs(state, test_dl, device)
    fold_probs.append(probs)
    if test_targets_cv is None:
        test_targets_cv = tgts
    print(f'  Fold {fold} OK  shape={probs.shape}')

# ── Estratégias de combinação ──────────────────────────────────────────
def decode(p): return (p > 0.5).sum(axis=1)

probs_mean  = np.mean(fold_probs, axis=0)
preds_mean  = decode(probs_mean)

preds_stack = np.vstack([decode(p) for p in fold_probs])  # (5, N)
# Majority vote
from scipy.stats import mode as smode
preds_vote  = smode(preds_stack, axis=0, keepdims=False).mode

results_cv = {}
for strat_name, preds in [('CV-Mean', preds_mean), ('CV-MajVote', preds_vote)]:
    acc   = accuracy_score(test_targets_cv, preds)
    kappa = cohen_kappa_score(test_targets_cv, preds, weights='quadratic')
    f1    = f1_score(test_targets_cv, preds, average='macro', zero_division=0)
    results_cv[strat_name] = {'acc': acc, 'kappa': kappa, 'f1': f1}
    print(f'\n{strat_name}:  QWK={kappa:.4f}  Acc={acc*100:.2f}%  F1={f1:.4f}')

# ── Relatório detalhado da melhor estratégia ───────────────────────────
best_strat = max(results_cv, key=lambda k: results_cv[k]['kappa'])
best_preds = preds_mean if best_strat == 'CV-Mean' else preds_vote
labels_isup = [f'ISUP {i}' for i in range(6)]

print(f'\nMelhor estratégia: {best_strat}')
print(classification_report(test_targets_cv, best_preds,
                             target_names=labels_isup, digits=4, zero_division=0))

cm      = confusion_matrix(test_targets_cv, best_preds)
cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.heatmap(cm,      annot=True, fmt='d',    cmap='Blues',
            xticklabels=labels_isup, yticklabels=labels_isup, ax=axes[0])
axes[0].set_title('Confusion Matrix (contagens)')
axes[0].set_ylabel('True'); axes[0].set_xlabel('Predicted')

sns.heatmap(cm_norm, annot=True, fmt='.3f', cmap='Blues',
            xticklabels=labels_isup, yticklabels=labels_isup, ax=axes[1])
axes[1].set_title('Confusion Matrix (normalizada)')
axes[1].set_ylabel('True'); axes[1].set_xlabel('Predicted')

plt.suptitle(f'ConvNeXt 5-Fold Ensemble ({best_strat})  QWK={results_cv[best_strat]["kappa"]:.4f}',
             fontsize=12)
plt.tight_layout()
plt.savefig('logs/convnext-cv-confusion-matrix.png', dpi=200, bbox_inches='tight')
plt.show()

# ── Salva resultados ───────────────────────────────────────────────────
log_cv = 'logs/convnext-cv-results.txt'
with open(log_cv, 'w') as f:
    f.write('ConvNeXt-Tiny — 5-Fold Cross-Validation\n')
    f.write('=' * 60 + '\n\n')
    f.write('Hiperparâmetros (Optuna best):\n')
    for k, v in bp.items():
        f.write(f'  {k}: {v}\n')
    f.write('\nResultados por fold (validação):\n')
    for r in fold_metrics:
        f.write(f'  Fold {r["fold"]}: QWK={r["best_kappa"]:.4f}  '
                f'Acc={r["best_acc"]*100:.2f}%  F1={r["best_f1"]:.4f}\n')
    f.write(f'  Mean: QWK={kappas.mean():.4f} ± {kappas.std():.4f}\n')
    f.write('\nTest set (ensemble dos 5 folds):\n')
    for strat, m in results_cv.items():
        f.write(f'  {strat}: QWK={m["kappa"]:.4f}  Acc={m["acc"]*100:.2f}%  F1={m["f1"]:.4f}\n')
    f.write('\nClassification Report:\n')
    f.write(classification_report(test_targets_cv, best_preds,
                                   target_names=labels_isup, digits=4, zero_division=0))
print(f'\nResultados salvos → {log_cv}')


## Plot Training History

## Evaluation on Test Set

## Summary

This notebook trained EfficientNet-B0 with:
1. **Entropy filtering** - Removed high-entropy samples for cleaner training data
2. **Ordinal loss** - Leveraged the ordered nature of ISUP grades (0 < 1 < 2 < 3 < 4 < 5)

The ordinal approach enforces the constraint that predictions should respect the natural ordering of grades, potentially improving performance on ordinal classification tasks like cancer grading.